# LP–BP and FB–BP: inspect the maps, hypotheses, and failure controls

**Question.** Do the proposed basis maps identify both chain boundaries, and which hypotheses are needed to reverse each correspondence?

**Paper locations.** Appendix F, Proposition 1 (“G-lifted products as balanced products”) and Proposition 2 (the fiber-bundle / balanced-product relation); Appendix E, Eq. (E9), fixes the nonabelian convention (`prop:lp-bp`, `prop:fb-bp`, `eq:noncomm-lp-checks`).

**Companion paper.** Yuan Yao, Guo Zhang, Yingli Yang, Yuanye Zhu, Xiaosi Xu, Xiao Yuan, and Ying Li, **“An introduction to qLDPC codes from a structural perspective.”** Submitted.

The paper numbers refer to the manuscript identified by `PAPER_SOURCE` in
[`paper_cases.py`](../qldpc_generator/paper_cases.py). Source labels are retained
to help locate the same material after renumbering. Equation numbers below are
local to this notebook; code outputs are computational results.

**Before running.** Follow the [installation instructions](../README.md#installation-and-execution),
select that Python environment, and run the cells in order. All matrix arithmetic
and ranks are over GF(2); matrix indices start at zero. For the coverage overview,
start with [06 — Paper validation](06-paper-validation.ipynb).

In [1]:
import numpy as np
from IPython.display import Markdown, display

def show_table(headers, rows):
    """Format results for reading; no mathematical calculations happen here."""
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## 1. Decide what would count as a correspondence

Equal N and K are insufficient. For each example, the returned `Equivalence`
has a BP code as `source` and an LP or FB code as `target`. It supplies maps
`p0`, `p1`, and `p2` from old basis indices to new ones. The associated
permutation matrices must satisfy both identities, using the same P1:

$$
H_X^{\rm target}=P_0H_X^{\rm BP}P_1^{\mathsf T},\qquad
H_Z^{\rm target}=P_2H_Z^{\rm BP}P_1^{\mathsf T}. \tag{1}
$$

The second identity is the transpose of the degree-two boundary identity.
We explicitly compute the residual matrices in Eq. (1); a successful check
has zero nonzero entries in each residual. The helper below only evaluates
these displayed equations and formats their results.

In [2]:
from qldpc_generator.algebra import (
    symmetric_group_3, group_algebra_matrix, permutation_matrix, cyclic_group,
)
from qldpc_generator.equivalences import (
    Equivalence, lp_to_bp, bp_to_lp, fb_to_bp, bp_to_fb,
)
from qldpc_generator import balanced_product

def boundary_residuals(eq):
    P0, P1, P2 = [permutation_matrix(p) for p in (eq.p0, eq.p1, eq.p2)]
    dx = (eq.target.hx + P0 @ eq.source.hx @ P1.T) % 2
    dz = (eq.target.hz + P2 @ eq.source.hz @ P1.T) % 2
    return int(np.count_nonzero(dx)), int(np.count_nonzero(dz))

## 2. LP–BP inputs: a genuinely nonabelian group

Use S3 with s = (12), t = (23), and multiplication acting by composition
with the right factor first. Library indices 0–5 mean e, s, t, st, ts, sts.
The coefficient tensors below encode the group-ring matrices

$$
A=\begin{pmatrix}e+s&e\end{pmatrix},\qquad
B=\begin{pmatrix}e+t\\s\end{pmatrix}\quad\text{over }\mathbb F_2[S_3]. \tag{2}
$$

For example, `[0,1]` denotes e+s; `[0]` denotes e. A acts by right
multiplication in its binary boundary, and B by left multiplication.
These boundary actions are distinct from the symmetry actions used to
take the balanced quotient.

In [3]:
group = symmetric_group_3()
names = ['e', 's', 't', 'st', 'ts', 'sts']
show_table(["Index", "Group element", "Images of points (0,1,2)"], [
    [i, names[i], tuple(group.labels[i])] for i in range(group.order)
])
assert group.mul(1, 2) != group.mul(2, 1)
print("s*t =", names[group.mul(1, 2)], "; t*s =", names[group.mul(2, 1)])
A = group_algebra_matrix(group, [[[0, 1], [0]]])
B = group_algebra_matrix(group, [[[0, 2]], [[1]]])
lp_bp, lp_eq = lp_to_bp(group, A, B)
show_table(["Object", "Size"], [
    ["A / B coefficient tensors (rows, columns, group order)", f"{A.shape} / {B.shape}"],
    ["Binary C / D boundaries", f"{lp_bp.dc.shape} / {lp_bp.dd.shape}"],
    ["BP H_X / H_Z", f"{lp_eq.source.hx.shape} / {lp_eq.source.hz.shape}"],
    ["LP H_X / H_Z", f"{lp_eq.target.hx.shape} / {lp_eq.target.hz.shape}"],
])
show_table(["Degree pair", "Number of quotient basis vectors"], [
    [degree, len(space.orbits)] for degree, space in lp_bp.spaces.items()
])

| Index | Group element | Images of points (0,1,2) |
| --- | --- | --- |
| 0 | e | (0, 1, 2) |
| 1 | s | (1, 0, 2) |
| 2 | t | (0, 2, 1) |
| 3 | st | (1, 2, 0) |
| 4 | ts | (2, 0, 1) |
| 5 | sts | (2, 1, 0) |

s*t = st ; t*s = ts


| Object | Size |
| --- | --- |
| A / B coefficient tensors (rows, columns, group order) | (1, 2, 6) / (2, 1, 6) |
| Binary C / D boundaries | (6, 12) / (12, 6) |
| BP H_X / H_Z | (12, 30) / (12, 30) |
| LP H_X / H_Z | (12, 30) / (12, 30) |

| Degree pair | Number of quotient basis vectors |
| --- | --- |
| (0, 0) | 12 |
| (0, 1) | 6 |
| (1, 0) | 24 |
| (1, 1) | 12 |

## 3. Derive and inspect the orbit map

For the convention of Proposition 1, the symmetry actions are
$(\alpha,x)\cdot u=(\alpha,u^{-1}x)$ and
$u\cdot(\beta,y)=(\beta,yu^{-1})$. Thus the balanced relation and map are

$$
((\alpha,x),(\beta,y))\sim((\alpha,u^{-1}x),(\beta,yu)),\qquad
\Phi([ (\alpha,x),(\beta,y) ])=(\alpha,\beta,yx). \tag{3}
$$

The product yx is unchanged along the orbit: (yu)(u⁻¹x) = yx. In contrast,
xy generally changes by conjugation. Here is one entire orbit, with the
outer indices suppressed. The assertions also check yx on **every** orbit
in all four degree pairs.

In [4]:
for space in lp_bp.spaces.values():
    for orbit in space.orbits:
        assert len({group.mul(d % 6, c % 6) for c, d in orbit}) == 1

sample = next(orbit for orbit in lp_bp.spaces[0, 0].orbits
              if len({group.mul(c % 6, d % 6) for c, d in orbit}) > 1)
show_table(["x", "y", "Correct product yx", "Incorrect product xy"], [
    [names[c % 6], names[d % 6], names[group.mul(d % 6, c % 6)],
     names[group.mul(c % 6, d % 6)]] for c, d in sample
])
assert len({group.mul(c % 6, d % 6) for c, d in sample}) > 1

| x | y | Correct product yx | Incorrect product xy |
| --- | --- | --- | --- |
| e | s | s | s |
| s | e | s | s |
| t | st | s | sts |
| st | sts | s | t |
| ts | t | s | sts |
| sts | ts | s | t |

The BP qubits are ordered (0,1) then (1,0), containing 6 and 24 orbits.
LP uses the opposite block order. For each representative, flatten the
target coordinates (alpha, beta, yx), then add the target block offset.
The table exposes the first few coordinates in each block. P0 and P2 use
the same orbit map in degrees (0,0) and (1,1).

In [5]:
rows = []
source_offset = 0
for degree in [(0, 1), (1, 0)]:
    space = lp_bp.spaces[degree]
    d_copies = lp_bp.dd.shape[degree[1]] // group.order
    target_offset = len(lp_bp.spaces[1, 0].orbits) if degree == (0, 1) else 0
    for j, (c, d) in enumerate(space.representatives[:4]):
        alpha, gx = divmod(c, group.order)
        beta, gy = divmod(d, group.order)
        k = group.mul(gy, gx)
        new = target_offset + (alpha*d_copies+beta)*group.order+k
        assert lp_eq.p1[source_offset+j] == new
        rows.append([degree, source_offset+j, (alpha, names[gx]),
                     (beta, names[gy]), names[k], new])
    source_offset += len(space.orbits)
show_table(["Degree", "BP index", "C representative", "D representative", "yx", "LP index"], rows)
assert lp_eq.verify() and boundary_residuals(lp_eq) == (0, 0)
A_back, B_back, reverse_lp = bp_to_lp(lp_bp)
assert np.array_equal(A_back, A) and np.array_equal(B_back, B)
assert boundary_residuals(reverse_lp) == (0, 0)
show_table(["Construction", "X-boundary residual nonzeros", "Z-boundary residual nonzeros"], [
    ["LP represented as BP", *boundary_residuals(lp_eq)],
    ["Group-ring inputs recovered from free BP", *boundary_residuals(reverse_lp)],
])

| Degree | BP index | C representative | D representative | yx | LP index |
| --- | --- | --- | --- | --- | --- |
| (0, 1) | 0 | (0, 'e') | (0, 'e') | e | 24 |
| (0, 1) | 1 | (0, 'e') | (0, 's') | s | 25 |
| (0, 1) | 2 | (0, 'e') | (0, 't') | t | 26 |
| (0, 1) | 3 | (0, 'e') | (0, 'st') | st | 27 |
| (1, 0) | 6 | (0, 'e') | (0, 'e') | e | 0 |
| (1, 0) | 7 | (0, 'e') | (0, 's') | s | 1 |
| (1, 0) | 8 | (0, 'e') | (0, 't') | t | 2 |
| (1, 0) | 9 | (0, 'e') | (0, 'st') | st | 3 |

| Construction | X-boundary residual nonzeros | Z-boundary residual nonzeros |
| --- | --- | --- |
| LP represented as BP | 0 | 0 |
| Group-ring inputs recovered from free BP | 0 | 0 |

### Failure control: default representatives can conceal the wrong order

Checking only the default representatives is not enough to detect this
particular mistake: their x coordinate is the identity, so xy and yx agree
there. The wrong formula is already disqualified by the full-orbit table
above. The next control additionally shows that its matrix identity fails
when different representatives are selected.

We deliberately build a candidate with xy in place of yx, preserving the
correct block ordering. With the first representatives it passes; with the
last representatives it fails. The correct map passes in both cases.

In [6]:
def wrong_xy_equivalence(bp, target):
    # Deliberately incorrect map used only as a failure control.
    maps = {}
    order = bp.group.order
    for degree, space in bp.spaces.items():
        d_copies = bp.dd.shape[degree[1]] // order
        maps[degree] = np.array([
            (c//order*d_copies+d//order)*order + bp.group.mul(c % order, d % order)
            for c, d in space.representatives
        ], dtype=int)
    p1 = np.r_[maps[0, 1]+len(maps[1, 0]), maps[1, 0]]
    return Equivalence(bp.code, target, maps[0, 0], p1, maps[1, 1], 'deliberately wrong xy')

rows = []
for choice in ('first', 'last'):
    bp, correct = lp_to_bp(group, A, B, representative=choice)
    wrong = wrong_xy_equivalence(bp, correct.target)
    assert correct.verify()
    assert wrong.verify() == (choice == 'first')
    rows.append([choice, correct.verify(), wrong.verify(), boundary_residuals(wrong)])
show_table(["Representatives", "yx passes", "xy passes", "xy residual nonzeros (X,Z)"], rows)

| Representatives | yx passes | xy passes | xy residual nonzeros (X,Z) |
| --- | --- | --- | --- |
| first | True | True | (0, 0) |
| last | True | False | (48, 60) |

## 4. FB–BP inputs: a fiber whose action is not free

Now use the two-term base and fiber boundaries, with one twist for each
nonzero base incidence:

$$
d_B=\begin{pmatrix}1&1\end{pmatrix},\qquad
d_F=\begin{pmatrix}1&1&1\end{pmatrix},\qquad
(\varphi(b_0,a),\varphi(b_1,a))=(s,st). \tag{4}
$$

The single F0 basis vector is fixed, while F1 carries the natural three-point
S3 action. Equivariance holds because permuting the columns of dF leaves it
unchanged. These actions are not free: the orbit sizes are 1 and 3, smaller
than the group order 6. Proposition 2 permits this.

The free cover has pairs (b,g) as basis vectors, with the right action on g.
Its boundary and the map to the twisted tensor product are

$$
\partial_{\widetilde B}(b,g)=\sum_{a\in\operatorname{supp}(\partial_B b)}
  (a,\varphi(b,a)g),\qquad
T([(b,g),f])=(b,g\cdot f). \tag{5}
$$

The balanced relation shifts u from the cover to the fiber, and
(gu)·f = g·(u·f), so T is independent of that representative choice.

In [7]:
base = np.array([[1, 1]], dtype=np.uint8)
fiber = np.array([[1, 1, 1]], dtype=np.uint8)
action0 = np.zeros((6, 1), dtype=int)
action1 = np.array(group.labels)
twists = np.array([[1, 3]])  # s, st in the table above.
for g in range(group.order):
    assert np.array_equal(fiber @ permutation_matrix(action1[g]) % 2,
                          permutation_matrix(action0[g]) @ fiber % 2)
fb_bp, fb_eq = fb_to_bp(base, fiber, group, action0, action1, twists)
show_table(["Object", "Shape or orbit size"], [
    ["Base boundary", base.shape], ["Cover boundary", fb_bp.dc.shape],
    ["Fiber boundary", fiber.shape],
    ["Orbit of an F0 basis vector", len(set(action0[:, 0]))],
    ["Orbit of an F1 basis vector", len(set(action1[:, 0]))],
    ["FB H_X / H_Z", f"{fb_eq.target.hx.shape} / {fb_eq.target.hz.shape}"],
])

| Object | Shape or orbit size |
| --- | --- |
| Base boundary | (1, 2) |
| Cover boundary | (6, 12) |
| Fiber boundary | (1, 3) |
| Orbit of an F0 basis vector | 1 |
| Orbit of an F1 basis vector | 3 |
| FB H_X / H_Z | (1, 5) / (6, 5) |

Trace T on one degree-(0,1) orbit. Although the group coordinate and fiber
index change, their combined image is constant. The code checks this
property for every orbit in every degree before evaluating both boundaries.

In [8]:
for (p, q), space in fb_bp.spaces.items():
    action = fb_bp.left[q]
    for orbit in space.orbits:
        images = {(c//group.order, int(action[c % group.order, f])) for c, f in orbit}
        assert len(images) == 1

sample = fb_bp.spaces[0, 1].orbits[0]
show_table(["Cover base index", "Group coordinate", "Fiber index f", "Image g.f"], [
    [c//6, names[c % 6], f, int(action1[c % 6, f])] for c, f in sample
])
base_back, twists_back, reverse_fb = bp_to_fb(fb_bp)
assert np.array_equal(base_back, base) and np.array_equal(twists_back, twists)
assert boundary_residuals(fb_eq) == boundary_residuals(reverse_fb) == (0, 0)
show_table(["Construction", "X-boundary residual nonzeros", "Z-boundary residual nonzeros"], [
    ["FB represented as BP", *boundary_residuals(fb_eq)],
    ["Base and twists recovered from covering BP", *boundary_residuals(reverse_fb)],
])

| Cover base index | Group coordinate | Fiber index f | Image g.f |
| --- | --- | --- | --- |
| 0 | e | 0 | 0 |
| 0 | s | 1 | 0 |
| 0 | t | 0 | 0 |
| 0 | st | 2 | 0 |
| 0 | ts | 1 | 0 |
| 0 | sts | 2 | 0 |

| Construction | X-boundary residual nonzeros | Z-boundary residual nonzeros |
| --- | --- | --- |
| FB represented as BP | 0 | 0 |
| Base and twists recovered from covering BP | 0 | 0 |

## 5. Test the hypotheses of the converses

| Direction | Hypotheses checked by this implementation |
| --- | --- |
| LP → BP | Group-ring inputs give equivariant boundaries and free actions on all four bases. |
| BP → LP | Both factors must have free actions in both degrees. |
| FB → BP | Fiber actions must be equivariant; they need not be free. The constructed cover is free. |
| BP → FB | The selected covering factor must be free in both degrees; its boundary must have at most one group label per base incidence for the chosen orbit representatives. |

The nonfree fiber above prevents the LP converse. A second example has a
free covering factor but two labels over one incidence, so it fails the
single-label FB converse. These are failures of the stated sufficient
hypotheses, not proofs that no alternative description can ever exist.

In [9]:
controls = []
try:
    bp_to_lp(fb_bp)
except ValueError as exc:
    assert 'free' in str(exc)
    controls.append(['Nonfree fiber passed to BP -> LP', str(exc)])
else:
    raise AssertionError('The LP converse must reject this nonfree action')

c2 = cyclic_group(2)
regular = c2.regular()
noncover = balanced_product(np.ones((2, 2), dtype=np.uint8), np.eye(2, dtype=np.uint8),
                            c2, (regular, regular), (regular, regular))
try:
    bp_to_fb(noncover)
except ValueError as exc:
    assert 'single-group-label' in str(exc)
    controls.append(['Two covering labels over one base incidence', str(exc)])
else:
    raise AssertionError('The FB converse must reject this covering boundary')
show_table(["Deliberately invalid converse input", "Expected refusal"], controls)

| Deliberately invalid converse input | Expected refusal |
| --- | --- |
| Nonfree fiber passed to BP -> LP | This correspondence requires free actions on every chosen basis |
| Two covering labels over one base incidence | Boundary violates the single-group-label covering hypothesis |

## 6. What this establishes

Both correspondence examples satisfy the complete matrix identities with a
shared qubit permutation. Their maps are constant on every orbit, the
reverse constructors recover the stated inputs, and the failure controls
expose multiplication-order and hypothesis errors.

These are finite checks of the maps in Propositions 1 and 2. The propositions'
general proofs remain in the paper. No code-distance claim follows just from
matching N and K; the full basis correspondence is the relevant evidence here.

Implementation: [basis maps and converse checks](../qldpc_generator/equivalences.py),
[orbit boundaries](../qldpc_generator/balanced.py), and
[tests including independently relabelled bases](../tests/test_products.py).
Continue with [05 — Bicycle and BB](05-bicycle-bb.ipynb).